# Phase 2 GBM-Based Aggregation Sensitivity and JRN Estimation on rel-f1

**What this does:** Re-runs Phase 2 aggregation sensitivity analysis using GBM probes (LightGBM) instead of MLP probes. For each join-task pair in the RelBench rel-f1 (Formula 1) relational database, this experiment:

1. Computes **JRN (Join Reproduction Number)** via GBM performance ratios across 5 aggregation strategies
2. Measures **aggregation strategy sensitivity** (CoV across mean/sum/max/std/all_combined)
3. Tests whether sensitivity peaks non-monotonically near JRN ~ 1 (**inverted-U hypothesis**) via quadratic and piecewise regression
4. Validates probe JRN against ground-truth JRN via Spearman correlation

This notebook loads pre-computed results from 65 join-task pairs (13 FK joins x 5 tasks) with 1365 LightGBM models and reproduces the statistical analysis and 6-panel visualization.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# statsmodels — NOT pre-installed on Colab, always install
_pip('statsmodels==0.14.6')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3',
         'matplotlib==3.10.0', 'seaborn==0.13.2')

In [ ]:
import json
import math
import os
import warnings
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import kruskal, spearmanr

warnings.filterwarnings("ignore")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-bc07ab-join-reproduction-number-epidemiology-in/main/experiment_iter3_phase_2_gbm_bas/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['datasets'][0]['examples'])} examples")
print(f"Metadata keys: {list(data['metadata'].keys())}")

## Configuration

Tunable parameters for the analysis. The original experiment used 3 seeds, 5 aggregation types, and 5 tasks across 13 FK joins (65 pairs total). These parameters control which subset to analyze.

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────
# Original values: AGG_TYPES has 5 types, piecewise search step=0.01
AGG_TYPES = ["mean", "sum", "max", "std", "all_combined"]
PIECEWISE_STEP = 0.01       # breakpoint search step (original: 0.01)
PIECEWISE_RANGE = (0.85, 1.25)  # breakpoint search range

## Step 1: Parse Examples into Per-Pair Details

Extract JRN, sensitivity (CoV), winning aggregation strategy, and per-aggregation performances from each join-task pair example.

In [ ]:
# Parse all examples from the loaded data
examples = data["datasets"][0]["examples"]
per_join_summary = data["metadata"]["per_join_summary"]

per_pair_details = []
for ex in examples:
    inp = json.loads(ex["input"])
    out = json.loads(ex["output"])
    per_pair_details.append({
        "join_idx": inp["join_idx"],
        "join_name": inp["join_name"],
        "fk_col": inp["fk_col"],
        "task_name": inp["task_name"],
        "jrn": out["jrn"],
        "sensitivity_cov": out["sensitivity_cov"],
        "baseline_perf": out["baseline_perf"],
        "best_agg_perf": out["best_agg_perf"],
        "winning_agg": out["winning_agg"],
        "gt_jrn": out["gt_jrn"],
        "agg_perfs": out["agg_perfs"],
    })

print(f"Parsed {len(per_pair_details)} join-task pair results")
print(f"Unique joins: {len(set(d['join_idx'] for d in per_pair_details))}")
print(f"Unique tasks: {sorted(set(d['task_name'] for d in per_pair_details))}")

## Step 2: Build JRN and Sensitivity Arrays

Construct parallel arrays of JRN values, sensitivity (CoV), probe JRN vs ground-truth JRN, and winning aggregation strategy for each pair.

In [ ]:
# Build arrays for statistical analysis
jrn_values = np.array([d["jrn"] for d in per_pair_details])
sens_values = np.array([d["sensitivity_cov"] for d in per_pair_details])

# Probe JRN vs Ground-Truth JRN
probe_jrns = [d["jrn"] for d in per_pair_details if d["gt_jrn"] is not None]
gt_jrns = [d["gt_jrn"] for d in per_pair_details if d["gt_jrn"] is not None]

# Winning aggregation per pair
winning_agg_dict = {(d["join_idx"], d["task_name"]): d["winning_agg"] for d in per_pair_details}

print(f"JRN stats: mean={np.mean(jrn_values):.3f}, std={np.std(jrn_values):.3f}, "
      f"min={np.min(jrn_values):.3f}, max={np.max(jrn_values):.3f}")
print(f"Sensitivity stats: mean={np.mean(sens_values):.3f}, std={np.std(sens_values):.3f}")
print(f"Probe-GT pairs available: {len(probe_jrns)}")

## Step 3: Probe-to-GT Correlation

Validate the lightweight GBM probe JRN estimates against ground-truth JRN (from larger GBM models with more estimators) using Spearman rank correlation.

In [ ]:
# Probe-to-GT Spearman correlation
if len(probe_jrns) >= 5:
    rho, rho_pval = spearmanr(probe_jrns, gt_jrns)
    print(f"Probe-GT Spearman rho={rho:.3f}, p={rho_pval:.4f} (n={len(probe_jrns)})")
else:
    rho, rho_pval = float("nan"), float("nan")
    print("Not enough common pairs for probe-GT correlation")

## Step 4: Statistical Analysis — Quadratic, Piecewise, and Non-parametric Tests

Test the **inverted-U hypothesis**: does aggregation sensitivity peak near JRN ~ 1?

- **Quadratic regression**: `sensitivity ~ beta0 + beta1*JRN + beta2*JRN^2`. Inverted-U requires beta2 < 0 (p < 0.05).
- **Piecewise linear**: find best breakpoint via BIC, check if slope changes from positive to negative.
- **Kruskal-Wallis**: non-parametric test across JRN tertiles.
- **Spearman monotonic**: JRN-sensitivity rank correlation.

In [ ]:
# ── 4a: Quadratic Model ──
beta0, beta1, beta2, p_beta2, r_sq_quad, peak_jrn = 0.0, 0.0, 0.0, 1.0, 0.0, None
try:
    X_quad = np.column_stack([jrn_values, jrn_values ** 2])
    X_quad = sm.add_constant(X_quad)
    model_quad = sm.OLS(sens_values, X_quad).fit()
    beta0, beta1, beta2 = model_quad.params
    p_beta2 = model_quad.pvalues[2] if len(model_quad.pvalues) > 2 else 1.0
    r_sq_quad = model_quad.rsquared
    if beta2 < 0:
        peak_jrn = -beta1 / (2 * beta2)
    print(f"Quadratic: beta0={beta0:.4f}, beta1={beta1:.4f}, beta2={beta2:.4f}, p={p_beta2:.4f}, R2={r_sq_quad:.4f}")
    if peak_jrn is not None:
        print(f"  Inverted-U peak at JRN={peak_jrn:.3f}")
    else:
        print(f"  beta2 > 0 => NOT inverted-U shaped")
except Exception as e:
    print(f"Quadratic fit failed: {e}")

# ── 4b: Piecewise Linear Model ──
best_bic, best_breakpoint = np.inf, 1.0
best_slope_before, best_slope_after = 0.0, 0.0
best_models_pw = (None, None)
try:
    for b in np.arange(PIECEWISE_RANGE[0], PIECEWISE_RANGE[1], PIECEWISE_STEP):
        mask_lo = jrn_values < b
        mask_hi = jrn_values >= b
        if mask_lo.sum() < 3 or mask_hi.sum() < 3:
            continue
        try:
            X_lo = sm.add_constant(jrn_values[mask_lo])
            X_hi = sm.add_constant(jrn_values[mask_hi])
            m_lo = sm.OLS(sens_values[mask_lo], X_lo).fit()
            m_hi = sm.OLS(sens_values[mask_hi], X_hi).fit()
            total_bic = m_lo.bic + m_hi.bic
            if total_bic < best_bic:
                best_bic = total_bic
                best_breakpoint = float(b)
                best_slope_before = float(m_lo.params[1]) if len(m_lo.params) > 1 else 0.0
                best_slope_after = float(m_hi.params[1]) if len(m_hi.params) > 1 else 0.0
                best_models_pw = (m_lo, m_hi)
        except Exception:
            continue
    print(f"Piecewise: breakpoint={best_breakpoint:.2f}, slope_before={best_slope_before:.4f}, "
          f"slope_after={best_slope_after:.4f}")
except Exception as e:
    print(f"Piecewise fit failed: {e}")

# ── 4c: Kruskal-Wallis test ──
kw_stat, kw_pval = float("nan"), float("nan")
try:
    tertile_edges = np.percentile(jrn_values, [33.3, 66.7])
    bins = np.digitize(jrn_values, tertile_edges)
    groups = [sens_values[bins == i] for i in range(3)]
    groups = [g for g in groups if len(g) >= 2]
    if len(groups) >= 2:
        kw_stat, kw_pval = kruskal(*groups)
        print(f"Kruskal-Wallis: H={kw_stat:.3f}, p={kw_pval:.4f}")
except Exception as e:
    print(f"Kruskal-Wallis failed: {e}")

# ── 4d: Monotonic check ──
mono_rho, mono_pval = float("nan"), float("nan")
try:
    mono_rho, mono_pval = spearmanr(jrn_values, sens_values)
    print(f"JRN-sensitivity Spearman: rho={mono_rho:.3f}, p={mono_pval:.4f}")
except Exception as e:
    print(f"Monotonic check failed: {e}")

# ── 4e: Winning agg by JRN bin ──
winning_agg_by_bin = {"low": {}, "mid": {}, "high": {}}
try:
    tertile_edges = np.percentile(jrn_values, [33.3, 66.7])
    for d in per_pair_details:
        jrn_val = d["jrn"]
        wagg = d["winning_agg"]
        if jrn_val < tertile_edges[0]:
            bn = "low"
        elif jrn_val < tertile_edges[1]:
            bn = "mid"
        else:
            bn = "high"
        winning_agg_by_bin[bn][wagg] = winning_agg_by_bin[bn].get(wagg, 0) + 1
    print(f"Winning agg by JRN bin: {json.dumps(winning_agg_by_bin, indent=2)}")
except Exception as e:
    print(f"Winning agg by bin failed: {e}")

# ── Conclusion ──
inverted_u_confirmed = bool(beta2 < 0 and p_beta2 < 0.05)
inverted_v_confirmed = bool(best_slope_before > 0 and best_slope_after < 0)

if inverted_u_confirmed or inverted_v_confirmed:
    conclusion = (f"CONFIRMS non-monotonic (inverted-U/V) relationship. "
                  f"Quadratic beta2={beta2:.4f} (p={p_beta2:.4f}).")
else:
    conclusion = (f"Does NOT confirm inverted-U shape. "
                  f"Quadratic beta2={beta2:.4f} (p={p_beta2:.4f}), monotonic rho={mono_rho:.3f}. "
                  f"Relationship appears {'monotonic' if abs(mono_rho) > 0.3 else 'flat/weak'}.")
print(f"\nConclusion: {conclusion}")

## Step 5: 6-Panel Visualization

Generate the main results figure with:
1. **JRN heatmap** (join x task)
2. **Sensitivity heatmap** (CoV across aggregation strategies)
3. **JRN vs Sensitivity scatter** + quadratic fit (threshold test)
4. **Probe vs GT JRN** scatter (validation)
5. **Winning aggregation by JRN bin** (bar chart)
6. **Piecewise linear fit** (breakpoint analysis)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# ── Plot 1: JRN heatmap ──
try:
    join_names = sorted(set(d["join_name"] for d in per_pair_details),
                        key=lambda x: min(d["join_idx"] for d in per_pair_details if d["join_name"] == x))
    task_short = sorted(set(d["task_name"] for d in per_pair_details))
    jrn_2d = np.full((len(join_names), len(task_short)), np.nan)
    join_idx_map = {name: i for i, name in enumerate(join_names)}
    task_idx_map = {name: i for i, name in enumerate(task_short)}
    for d in per_pair_details:
        if d["join_name"] in join_idx_map and d["task_name"] in task_idx_map:
            jrn_2d[join_idx_map[d["join_name"]], task_idx_map[d["task_name"]]] = d["jrn"]
    jrn_heatmap_df = pd.DataFrame(jrn_2d, index=join_names, columns=task_short)
    ax1 = axes[0, 0]
    sns.heatmap(jrn_heatmap_df, annot=True, fmt=".2f", cmap="RdYlGn", center=1.0, ax=ax1,
                annot_kws={"size": 7}, cbar_kws={"shrink": 0.8})
    ax1.set_title("JRN Matrix", fontsize=11)
    ax1.tick_params(axis="both", labelsize=7)
except Exception as e:
    print(f"JRN heatmap failed: {e}")

# ── Plot 2: Sensitivity heatmap ──
try:
    sens_2d = np.full((len(join_names), len(task_short)), np.nan)
    for d in per_pair_details:
        if d["join_name"] in join_idx_map and d["task_name"] in task_idx_map:
            sens_2d[join_idx_map[d["join_name"]], task_idx_map[d["task_name"]]] = d["sensitivity_cov"]
    sens_heatmap_df = pd.DataFrame(sens_2d, index=join_names, columns=task_short)
    ax2 = axes[0, 1]
    sns.heatmap(sens_heatmap_df, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax2,
                annot_kws={"size": 7}, cbar_kws={"shrink": 0.8})
    ax2.set_title("Aggregation Sensitivity (CoV)", fontsize=11)
    ax2.tick_params(axis="both", labelsize=7)
except Exception as e:
    print(f"Sensitivity heatmap failed: {e}")

# ── Plot 3: JRN vs Sensitivity scatter + quadratic ──
try:
    ax3 = axes[0, 2]
    ax3.scatter(jrn_values, sens_values, alpha=0.6, c="steelblue", edgecolors="k", s=40)
    jrn_sorted = np.linspace(jrn_values.min() - 0.05, jrn_values.max() + 0.05, 100)
    quad_pred = beta0 + beta1 * jrn_sorted + beta2 * jrn_sorted ** 2
    ax3.plot(jrn_sorted, quad_pred, "r-", lw=2,
             label=f"Quad (beta2={beta2:.3f}, p={p_beta2:.3f})")
    ax3.axvline(x=1.0, color="gray", linestyle="--", alpha=0.5, label="JRN=1")
    if peak_jrn is not None:
        ax3.axvline(x=peak_jrn, color="orange", linestyle=":", alpha=0.7,
                    label=f"Peak={peak_jrn:.2f}")
    ax3.set_xlabel("JRN")
    ax3.set_ylabel("Sensitivity (CoV)")
    ax3.set_title("Phase 2: Threshold Test", fontsize=11)
    ax3.legend(fontsize=8)
except Exception as e:
    print(f"Scatter plot failed: {e}")

# ── Plot 4: Probe vs GT JRN ──
try:
    ax4 = axes[1, 0]
    if probe_jrns and gt_jrns:
        ax4.scatter(probe_jrns, gt_jrns, alpha=0.6, c="steelblue", edgecolors="k", s=40)
        lims = [min(min(probe_jrns), min(gt_jrns)), max(max(probe_jrns), max(gt_jrns))]
        ax4.plot(lims, lims, "r--", alpha=0.5)
        ax4.set_xlabel("Probe JRN")
        ax4.set_ylabel("GT JRN")
        ax4.set_title(f"Probe-GT (rho={rho:.3f})", fontsize=11)
    else:
        ax4.text(0.5, 0.5, "No GT data", ha="center", va="center", transform=ax4.transAxes)
except Exception as e:
    print(f"Probe-GT plot failed: {e}")

# ── Plot 5: Winning agg by JRN bin ──
try:
    ax5 = axes[1, 1]
    bin_labels = ["low", "mid", "high"]
    bar_data = [[winning_agg_by_bin.get(bl, {}).get(a, 0) for a in AGG_TYPES] for bl in bin_labels]
    bar_df = pd.DataFrame(bar_data, index=bin_labels, columns=AGG_TYPES)
    bar_df.plot(kind="bar", ax=ax5, colormap="Set2")
    ax5.set_title("Winning Agg by JRN Bin", fontsize=11)
    ax5.set_ylabel("Count")
    ax5.legend(fontsize=7, ncol=2)
    ax5.tick_params(axis="x", rotation=0)
except Exception as e:
    print(f"Winning agg plot failed: {e}")

# ── Plot 6: Piecewise linear fit ──
try:
    ax6 = axes[1, 2]
    ax6.scatter(jrn_values, sens_values, alpha=0.6, c="steelblue", edgecolors="k", s=40)
    ax6.axvline(x=best_breakpoint, color="red", linestyle="--", lw=2,
                label=f"Break={best_breakpoint:.2f}")
    if best_models_pw[0] is not None and best_models_pw[1] is not None:
        jrn_lo = np.linspace(jrn_values.min(), best_breakpoint, 50)
        jrn_hi = np.linspace(best_breakpoint, jrn_values.max(), 50)
        pred_lo = best_models_pw[0].predict(sm.add_constant(jrn_lo))
        pred_hi = best_models_pw[1].predict(sm.add_constant(jrn_hi))
        ax6.plot(jrn_lo, pred_lo, "g-", lw=2, label=f"slope={best_slope_before:.3f}")
        ax6.plot(jrn_hi, pred_hi, "m-", lw=2, label=f"slope={best_slope_after:.3f}")
    ax6.set_xlabel("JRN")
    ax6.set_ylabel("Sensitivity (CoV)")
    ax6.set_title("Piecewise Linear Fit", fontsize=11)
    ax6.legend(fontsize=8)
except Exception as e:
    print(f"Piecewise plot failed: {e}")

plt.tight_layout()
plt.savefig("jrn_phase2_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figure: jrn_phase2_results.png")

## Results Summary

Key findings from the Phase 2 GBM-based analysis across all join-task pairs.

In [ ]:
# ── Per-Join Summary Table ──
print("=" * 90)
print("PER-JOIN SUMMARY")
print("=" * 90)
print(f"{'Join':<40} {'Fanout':>8} {'Mean JRN':>10} {'Std JRN':>9} {'Mean Sens':>10} {'N Tasks':>8}")
print("-" * 90)
for js in per_join_summary:
    print(f"{js['join_name']:<40} {js['fanout_mean']:>8.1f} {js['mean_jrn']:>10.4f} "
          f"{js['std_jrn']:>9.4f} {js['mean_sensitivity']:>10.4f} {js['n_tasks']:>8}")

# ── Key Statistics ──
print("\n" + "=" * 90)
print("KEY STATISTICS")
print("=" * 90)
print(f"Total join-task pairs analyzed:  {len(per_pair_details)}")
print(f"Total models trained:            {data['metadata']['num_total_models_trained']}")
print(f"JRN range:                       {np.min(jrn_values):.3f} - {np.max(jrn_values):.3f} (mean {np.mean(jrn_values):.3f})")
print(f"Mean sensitivity (CoV):          {np.mean(sens_values):.3f}")
print(f"")
print(f"Probe-GT Spearman rho:           {rho:.3f} (p={rho_pval:.4f})")
print(f"Quadratic beta2:                 {beta2:.4f} (p={p_beta2:.4f})")
print(f"Piecewise breakpoint:            {best_breakpoint:.2f} (slope_before={best_slope_before:.4f}, slope_after={best_slope_after:.4f})")
print(f"Kruskal-Wallis:                  H={kw_stat:.3f} (p={kw_pval:.4f})")
print(f"JRN-sensitivity Spearman:        rho={mono_rho:.3f} (p={mono_pval:.4f})")
print(f"")
print(f"Inverted-U confirmed:            {inverted_u_confirmed}")
print(f"Inverted-V confirmed:            {inverted_v_confirmed}")
print(f"")
print(f"CONCLUSION: {conclusion}")